In [ ]:
import sys
from pathlib import Path
import numpy as np

src_path = Path.cwd().parent / "src"
sys.path.append(str(src_path.resolve()))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split        # from scikit-learn lib

from plotting import plot_histogram
from config import PATH_AMESHOUSING

In [ ]:
# READ THE CSV DATA
df = pd.read_csv(PATH_AMESHOUSING)

# Step 1 - Explore and understand the data

In [ ]:
# the available variables and their data types
df.info()

In [ ]:
# the distribution of SalePrice
print(df["SalePrice"].describe())

print(df["SalePrice"].plot(kind="hist"))

In [ ]:
# missing values
df.isna().sum().sort_values(ascending=False)

In [ ]:
# Fillip's code:
NanValues = df.isnull().sum()
NanValues = NanValues[NanValues > 0].sort_values(ascending=False)

plt.figure(figsize=(12, 6))
bars = plt.bar(NanValues.index, NanValues.values)

plt.yscale('log')
plt.xticks(rotation=45)
plt.ylabel("Number of missing values (log scale)")
plt.xlabel("Column")
plt.title("Missing Values per Column")
plt.grid()

# Add the actual count on top of each bar
for bar, value in zip(bars, NanValues.values):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
              str(value), ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# checking values of categories showing signs of having outliers when using graph of 'missing values per column' (Fillip's code)
print(df["Pool QC"].value_counts()) # little useful
print(df["Misc Feature"].value_counts()) 
print(df["Alley"].value_counts()) 
print(df["Fence"].value_counts()) 
print(df["Mas Vnr Type"].value_counts()) 
print(df["Fireplace Qu"].value_counts()) 
print(df["Lot Frontage"].value_counts()) 
print(df["Garage Qual"].value_counts()) 
print(df["Garage Cond"].value_counts()) 
print(df["Garage Yr Blt"].value_counts()) 
print(df["Garage Finish"].value_counts())
print(df["Garage Type"].value_counts()) 
print(df["Bsmt Exposure"].value_counts()) 
print(df["BsmtFin Type 2"].value_counts()) 
print(df["Bsmt Cond"].value_counts())
print(df["Bsmt Qual"].value_counts()) 
print(df["BsmtFin Type 1"].value_counts()) 

In [ ]:
# unusual values or patterns

df.describe().T
# The max value being much bigger than the 75% value indicates an outlier

In [ ]:
# checking values of categories showing signs of having outliers when using df.describe()
print(df["Low Qual Fin SF"].value_counts()) # little useful
print(df["TotRms AbvGrd"].value_counts())   # useful
print(df["Wood Deck SF"].value_counts())    # little useful
print(df["Open Porch SF"].value_counts())   # little useful
print(df["Enclosed Porch"].value_counts())  # little useful
print(df["3Ssn Porch"].value_counts())      # little useful
print(df["Screen Porch"].value_counts())    # little useful
print(df["Pool Area"].value_counts())       # little useful
print(df["Misc Val"].value_counts())        # little useful
print(df["Mo Sold"].value_counts())         # useful

In [ ]:
# relationships that may be useful for predicting house prices

# Heatmap for correlation (inspired by Kacper's code)
target_to_compare_to = df.corr(numeric_only = True)["SalePrice"].sort_values(ascending = False)

plt.figure(figsize = (3, 10))
sns.heatmap(target_to_compare_to.to_frame(), cmap = "coolwarm", annot = True, cbar = True)

In [ ]:
# a bit more visual, easier to read
plt.figure(figsize=(8, 10))
target_to_compare_to.drop("SalePrice").plot(kind="barh")
plt.title("Correlation with SalePrice")
plt.xlabel("Correlation coefficient")

# Step 2 - Split onto training and test sets

In [ ]:
# splitting df into training and testing data
# test_size = 0.2 means 20% is held out for testing
# random_state = 13 means we use the same split every time we run notebook. makes it reproduceable(we can do the same again and again)
train_df, test_df = train_test_split(df, test_size = 0.2, random_state = 13)

print(train_df.shape, test_df.shape)

# from here on, we will use train_df for all exploration/cleaning

# Step 3 - Prepare the data

In [ ]:
# examines what categories has highest correllation to SalePrice in test data
train_target_to_compare_to = train_df.corr(numeric_only = True)["SalePrice"].sort_values(ascending = False)

plt.figure(figsize = (3, 10))
sns.heatmap(train_target_to_compare_to.to_frame(), cmap = "coolwarm", annot = True, cbar = True)

In [ ]:
# Finds out how many missing values each category in the test data has
# inspired by Fillip's code:
train_NanValues = train_df.isnull().sum()
train_NanValues = train_NanValues[train_NanValues > 0].sort_values(ascending=False)

plt.figure(figsize=(12, 8))
bars = plt.bar(train_NanValues.index, train_NanValues.values)

plt.yscale('log')
plt.xticks(rotation=45)
plt.ylabel("Number of missing values in training data(log scale)")
plt.xlabel("Column")
plt.title("Missing Values per Column in training data")
plt.grid()

# Add the actual count(with percentage equivalent) on top of each bar
total_rows = len(train_df)
for bar, value in zip(bars, train_NanValues.values):
    pct = value / total_rows * 100
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
              f"{value}\n({pct:.1f}%)", ha='center', va='bottom', fontsize=8)
    

plt.tight_layout()
plt.show()

In [ ]:
# colums we are considering dropping (based on it missing more than 15% of the data):
cols_to_drop = ["Pool QC",
                "Misc Feature",
                "Alley",
                "Fence",
                "Mas Vnr Type",
                "Fireplace Qu",
                "Lot Frontage",     # numeric value, correlation to SalePrice = 0.36  
                ]

In [ ]:
# checking for outliers:
pd.options.display.float_format = "{:,.2e}".format
test_df.describe().T